# Tool-Invoking Chatbots in Real Time
Companion notebook for the **Tool-Invoking Chatbots in Real Time** section of Chapter 11.

Pattern: **understand -> call tool -> confirm if needed -> return result**. Trigger tools from both intent *and* context -- "cancel my subscription" is an action, not just a reply, so it needs identity and confirmation checks before it runs.

Small stand-ins for `detect_intent` and `order_api`, so the book's `handle_request` is actually callable:

In [1]:
def detect_intent(user_msg: str) -> str:
    text = user_msg.lower()
    if "order" in text or "where is my" in text:
        return "check_order"
    if "cancel" in text and "subscription" in text:
        return "cancel_subscription"
    return "unknown"


class OrderAPI:
    def get_status(self, order_id: str) -> dict:
        return {"order_id": order_id, "status": "out for delivery"}


order_api = OrderAPI()

This is the book's `handle_request`, unchanged:

In [2]:
def handle_request(user_msg, context):
    intent = detect_intent(user_msg)

    if intent == "check_order":
        result = order_api.get_status(context["order_id"])
        return f"Order {context['order_id']} is {result['status']}."

    if intent == "cancel_subscription":
        if not context.get("verified"):
            return "Please verify your identity first."
        return "I can cancel it. Do you want me to continue?"

    return "I can help with status checks, updates, and support actions."

In [3]:
print(handle_request("Where is my order?", {"order_id": "O-4471"}))
print(handle_request("Cancel my subscription please", {"verified": False}))
print(handle_request("Cancel my subscription please", {"verified": True}))
print(handle_request("Tell me a joke", {}))

Order O-4471 is out for delivery.
Please verify your identity first.
I can cancel it. Do you want me to continue?
I can help with status checks, updates, and support actions.


Notice `cancel_subscription` never executes the cancellation in one turn -- it verifies identity, then asks for confirmation. The actual cancel only happens on an explicit follow-up, which is exactly the **confirm if needed** step for a hard-to-undo action.

**Failure handling**: tools time out, return partial data, or fail validation -- say what happened, whether it retried, and what the user can do next, instead of hiding it:

In [4]:
class ToolError(Exception):
    def __init__(self, attempts: int):
        self.attempts = attempts


def call_billing_tool_that_fails(*_args, **_kwargs):
    raise ToolError(attempts=2)


def call_with_visible_failure(tool_call, *args, max_attempts=2, **kwargs) -> str:
    try:
        return str(tool_call(*args, **kwargs))
    except ToolError as e:
        return (
            f"I could not reach the billing system right now. I tried {e.attempts} time(s). "
            "Please try again in a minute, or I can create a support ticket."
        )


print(call_with_visible_failure(call_billing_tool_that_fails, "U123"))

I could not reach the billing system right now. I tried 2 time(s). Please try again in a minute, or I can create a support ticket.
